# Data Inspection

This notebook profiles the car dataset before analysis or modeling. It checks the dataset structure, data quality, distributions, categorical values, possible outliers, and the first derived feature used for later analysis.

The notebook keeps the original data in `df` until the cleaning step, then creates `df_clean` without exact duplicate rows.

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path


def iter_start_paths():
    yield Path.cwd()

    session_name = os.environ.get("JPY_SESSION_NAME")
    if session_name:
        session_path = Path(session_name)
        if not session_path.is_absolute():
            session_path = (Path.cwd() / session_path).resolve()
        yield session_path.parent if session_path.suffix == ".ipynb" else session_path

    ipython_shell = globals().get("get_ipython", lambda: None)()
    if ipython_shell is not None:
        vscode_notebook = ipython_shell.user_ns.get("__vsc_ipynb_file__")
        if vscode_notebook:
            yield Path(vscode_notebook).resolve().parent


def find_project_root() -> Path:
    visited = set()
    for start_path in iter_start_paths():
        for candidate in [start_path, *start_path.parents]:
            if candidate in visited:
                continue
            visited.add(candidate)
            raw_data_file = candidate / "data" / "CAR DETAILS FROM CAR DEKHO.csv"
            if raw_data_file.exists() and (candidate / "notebooks").exists():
                return candidate
    raise FileNotFoundError("Could not locate project root containing data/CAR DETAILS FROM CAR DEKHO.csv")


PROJECT_ROOT = find_project_root()
RAW_DATA_PATH = PROJECT_ROOT / "data" / "CAR DETAILS FROM CAR DEKHO.csv"
CLEAN_DATA_PATH = PROJECT_ROOT / "data" / "cleaned_car_data.csv"

df = pd.read_csv(RAW_DATA_PATH)
df.head()


## 1. Load the Dataset

The dataset is loaded into `df`. The first preview confirms that the file was read successfully and gives an initial view of the available columns and records.

In [ ]:
df.shape

In [ ]:
df.info()

## 2. Inspect Dataset Structure and Quality

These checks establish the number of rows and columns, the data types, and whether missing values or exact duplicate records are present. This is the core data-quality review before any transformation.

In [ ]:
df.isnull().any()

In [ ]:
df.duplicated().any()

In [ ]:
df.duplicated().sum()

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

In [ ]:
duplicated_rows = df[df.duplicated(keep = False)]
duplicated_rows.head()

In [ ]:
df.drop_duplicates().shape

## 3. Review Categorical Values

Value counts show the distinct categories and their frequencies for fuel type, seller type, transmission, and owner history. This helps identify unexpected labels and imbalanced categories.

In [ ]:
categorical_columns = [
    "fuel",
    "seller_type",
    "transmission",
    "owner"
]

for column in categorical_columns:
    print(f"\n{column}:")
    print(df[column].value_counts())

## 4. Explore Distributions and Potential Outliers

The histogram view and summary tables provide a quick sense of the numerical ranges, skew, and unusual records. The high-mileage review is an inspection step rather than an automatic decision to remove those observations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

numeric_columns = ["year", "selling_price", "km_driven"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for column, ax in zip(numeric_columns, axes):
    sns.histplot(data=df, x=column, bins=30, kde=True, ax=ax)
    ax.set_title(f"Distribution of {column}")

plt.tight_layout()
plt.show()

In [ ]:
df["owner"].value_counts()

In [ ]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Rows after removing exact duplicates:", df.drop_duplicates().shape[0])

In [ ]:
df[["year", "selling_price", "km_driven"]].agg(
    ["min", "max", "mean", "median"]
).round(2)

In [ ]:
df.nlargest(10, "selling_price")[
    ["name", "year", "selling_price", "km_driven", "fuel", "owner"]
]

In [ ]:
df.nlargest(10, "km_driven")[
    ["name", "year", "selling_price", "km_driven", "fuel", "owner"]
]

## 5. Create the Clean Working Copy

After reviewing the original records, the notebook creates `df_clean` by removing exact duplicates. Keeping `df_raw` preserves a reference to the original row count and makes the cleaning impact transparent.

In [ ]:
# Keep the original dataset unchanged so we can compare it with the cleaned version later
df_raw = df.copy()
# Create a working copy without exact duplicate rows
df_clean = df.drop_duplicates().copy()
print("Original rows:", len(df_raw))
print("Rows after removing duplicates:", len(df_clean))
print("Duplicates remaining:", df_clean.duplicated().sum())

In [ ]:
# Inspect records with unusually high mileage
df_clean[df_clean["km_driven"] > 300000][
    ["name", "year", "selling_price", "km_driven", "owner"]
].sort_values("km_driven", ascending=False)

In [ ]:
print("Oldest manufacturing year:", df_clean["year"].min())
print("Newest manufacturing year:", df_clean["year"].max())

df_clean["year"].value_counts().sort_index()

## 6. Derive Car Age for Later Analysis

The newest manufacturing year in the cleaned data is used as a reference point for a simple `car_age` feature. The following checks confirm its range and whether the derivation introduced missing values.

In [ ]:
REFERENCE_YEAR = df_clean["year"].max()

df_clean["car_age"] = REFERENCE_YEAR - df_clean["year"]

df_clean[["name", "year", "car_age"]].head(10)

In [ ]:
print("Reference year:", REFERENCE_YEAR)
print("Minimum car age:", df_clean["car_age"].min())
print("Maximum car age:", df_clean["car_age"].max())
print("Missing car ages:", df_clean["car_age"].isnull().sum())

## 7. Check a Basic Relationship

The correlation between car age and selling price is included as an initial diagnostic. It can guide the deeper relationship analysis in the separate exploratory-analysis notebook, but it does not establish causation.

In [ ]:
df_clean[["car_age", "selling_price"]].corr()

## 8. Export the Inspected Dataset

The cleaned dataset, including the derived `car_age` column, is written to a CSV file for reuse in the next notebook. The original `df` is not overwritten.

In [ ]:
df_clean.to_csv(CLEAN_DATA_PATH, index=False)
print(f"Cleaned dataset saved to: {CLEAN_DATA_PATH}")
